# Figures and tables: parse, enrich, or embed the page

Compare three ways of making chart and table pages answerable (parsing to text, enriching the parse
with a vision model, and embedding the page image) on 1,000 FinanceBench filing pages, with one reader
model and the page as the unit.

You will:

- Build the case first: questions whose answer is printed only on a chart, and how often a chart's
  values are also in the page text.
- Run eight arms and four oracle rows that differ only in what is indexed and what the reader sees.
- Split each arm's loss into retrieval loss and reading loss.
- Price each approach under a scenario, indexing cost against query cost and local against paid.

The approaches fall into five families:

| Family | What is indexed | What reads at answer time | Where it pays |
| --- | --- | --- | --- |
| 1. Text-only parse | Docling's text layer, table-aware chunks | A text model over chunks | Almost nothing; the parse is CPU time |
| 2. Parse plus enrichment (the harness) | Family 1 plus vision-model descriptions of pictures and summaries of tables, as text | A text model over chunks | One vision call per element, at indexing and at every re-index |
| 3. Page-image retrieval, text reader | ColModernVBERT page embeddings | A text model over the retrieved pages' text | Page embedding at indexing; a query embedding per question |
| 4. Page-image retrieval, image reader | The same embeddings | A vision model over the page images | Image tokens on every question |
| 5. Hybrids | Both | Images and text, or text retrieval with an image reader | Both sides |

The rows below use these letters. `E1` and `E2` are the two enrichment prompts, compared in their own section.

| Row | Retrieval | Reader input | Family |
| --- | --- | --- | --- |
| `T` | `bge-small-en-v1.5` over Docling text, table-aware chunks | Top 5 chunks | 1. Text-only parse |
| `E1` | As `T`, with generic picture descriptions inlined | Top 5 chunks | 2. Parse plus enrichment |
| `E2` | As `T`, with domain-prompted descriptions and table summaries inlined | Top 5 chunks | 2. Parse plus enrichment |
| `L` | ColBERT over `E2`'s chunks | Top 5 chunks | Late-interaction control |
| `VT` | ColModernVBERT page images | Top 3 pages' `E2` text | 3. Page retrieval, text reader |
| `VV` | ColModernVBERT page images | Top 3 page images | 4. Page retrieval, image reader |
| `TV` | `E2`'s ranking, as pages | Top 3 page images | 5. Hybrid |
| `H` | Reciprocal rank fusion of `E2` and `VV` | Top 3 pages, image and `E2` text | 5. Hybrid |
| `O-text`, `O-enriched`, `O-image`, `O-both` | The gold page | As named | Oracles |

Two things compare one to one across families: answer accuracy on the same questions with the same
reader model, and page-level retrieval over the same page universe. Chunk metrics and raw cost profiles
do not, and nothing below ranks them across families.

## Choose your models and the run mode

`RUN_EXPENSIVE = False` reads the reference run: every model step was built on Vector's on-prem models
(`Qwen3_8-27B` read pictures, wrote questions, and answered for every row; `Qwen3_8-Flash-Next` checked
the questions; `Qwen3_6-35B-A3B` judged answers that no number settled). Each cell labels what it read
and what rebuilding it on your model would cost.

`RUN_EXPENSIVE = True` answers a 60-question stratified subset with your generation model in every row,
so your rows compare with each other. Your rows do not compare with the reference rows, because the
reader differs.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd()))

from aieng.advanced_rag import spend
from aieng.advanced_rag.corpora import financebench
from aieng.advanced_rag.models import choose_models
from vdocs import arms, report
from vdocs.pages import page_id
from vdocs.pipeline import Pipeline
from vdocs.scoring import Scenario


RUN_EXPENSIVE = False
# Re-describe pictures and re-summarise tables with your extraction model. Off
# keeps the reference descriptions, so only the reader changes between runs.
REBUILD_ENRICHMENT = False
# Print question text, descriptions, and page images. FinanceBench is licensed
# CC BY-NC 4.0, so the committed outputs carry identifiers and metrics only.
SHOW_CORPUS_TEXT = False

MODELS = choose_models(show=True)
WHICH = "subset" if RUN_EXPENSIVE else "all"
pipe = Pipeline(MODELS, live=RUN_EXPENSIVE, rebuild_enrichment=REBUILD_ENRICHMENT)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)


## What the filings contain

The page universe holds every page with a picture over a tenth of the page in the 84 FinanceBench
filings, a sample of pages with a complex table, the gold pages of FinanceBench's own questions on
those filings, and distractor pages from the same filings. Every arm indexes these 1,000 pages and no
others.

In [2]:
universe = pipe.universe()
counts = report.selection_counts(pipe)
print(f"page universe  {len(universe.pages):,} pages from {len(universe.doc_ids)} filings")
for name, group in universe.to_dict().items():
    print(f"  {name:<12}{len(group):>5}")
print(f"pictures {counts['pictures']}, of which {counts['pictures_described']} cover 5% of a page or more")
print(
    f"tables   {counts['tables']}, of which {counts['tables_summarised']} are complex (6+ columns or a spanning header)"
)

figure_pages = set(universe.figure)
queries = financebench.load_queries()
on_figures = [q for q in queries if any(page_id(q.doc_id, p) in figure_pages for p in q.gold_pages)]
print(f"FinanceBench questions with a gold page in the figure group: {len(on_figures)} of {len(queries)}")
if SHOW_CORPUS_TEXT:
    import IPython.display
    from vdocs.pages import render_page

    for identifier in universe.figure[:6]:
        IPython.display.display(render_page(pipe.corpus, identifier, scale=0.4))

page universe  1,000 pages from 49 filings
  figure        103
  table         100
  control        98
  distractor    699
pictures 144, of which 117 cover 5% of a page or more
tables   823, of which 530 are complex (6+ columns or a spanning header)
FinanceBench questions with a gold page in the figure group: 1 of 150


One of FinanceBench's 150 questions has a gold page in the figure group, so the benchmark says almost nothing about charts. The filings hold charts in useful numbers: performance graphs, revenue and margin bars, segment pies, value-at-risk histograms, and full-page infographics. Some of those infographics are images with no text layer at all.

## Build the question set

FinanceBench barely tests figures, so the questions are built with the package's synthetic design,
adapted for a picture region as evidence. For each figure page, `Qwen3_8-27B` sees the page and crops of
its large pictures and writes up to two statements read off a chart, with a question for each. The
statement must name the chart element it reads from, and that name must appear in the page text or the
picture's description. `Qwen3_8-Flash-Next` then answers each question from the page image and from the
page text alone:

- A question it cannot answer from the image is dropped.
- A question it answers from the image and not from the text is figure-only.
- A question it answers from both is duplicated: the chart's values are also in the page's text layer.

Table questions take one numeric cell of a complex table with its row label and column header. The
checker must answer them from the page text, and each is tagged `split` when a plain 1,200-character
window separates the cell from its header. Controls are FinanceBench questions on the same filings with
their human answers. Nulls are near-miss rewrites that neither the gold page image nor the top pages
from BM25 and dense retrieval answer.

In [3]:
question_set = pipe.question_set()
summary = report.question_summary(pipe)
print()
for group in report.GROUPS:
    print(f"{group:<13}{summary['counts'].get(group, 0):>5}")
print(f"table questions split by plain chunking: {summary['table_split']}")
print()
print(f"{'kind':<8}{'stage':<46}{'entered':>8}{'passed':>8}")
for row in summary["funnel"]:
    print(f"{row['kind']:<8}{row['stage']:<46}{row['entered']:>8}{row['passed']:>8}")
if SHOW_CORPUS_TEXT:
    for q in pipe.questions()[:10]:
        print(q["query_id"], q["group"], q.get("subgroup"), q["gold_pages"], q["question"], "->", q["answer"])

picture-descriptions reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
                     117 calls, 72,659 prompt + 20,593 completion tokens
picture-descriptions reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
                     117 calls, 99,701 prompt + 38,773 completion tokens
table-summaries reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
                530 calls, 459,609 prompt + 99,704 completion tokens
question-set reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
             715 calls, 962,098 prompt + 38,164 completion tokens

figure-only     76
duplicated      44
table           60
control         40
null            26
table questions split by plain chunking: 16

kind    stage                                          entered  passed
figure  page yields a chart question                       103      97
figure  one numeric answer                                 148     148
figure  chart elem

The funnel kept 120 of 148 chart questions. 76 are figure-only and 44 duplicated: in more than a third of the questions the checker could answer, the chart's values were also in Docling's text of the page, mostly because born-digital charts print their labels as text and performance graphs carry a companion table. Eleven candidates failed the region check and the checker could not answer 15 more from the image; both were dropped, and the 122 left were capped at 120. Table questions passed at 88 of 96, and 26 of 50 near-miss nulls survived both absence checks.

A hand check of 10 sampled questions against their pages found the reference answer correct in all 10. One figure question words a waterfall bar (a year-over-year change) as an amount, and one near-miss null changes the filing year rather than the period.

A figure question survived only if a model read it off the page image, so the oracle rows on figure questions sit high by construction, and the reference answers were generated and checked by models, not by people.

## Enrich the parse: the harness approach

The harness parses each filing with Docling, sends selected elements to a vision model, and indexes
what comes back as text, so the dense index, a BM25 index, and the reader all consume it. Two prompts
are compared on the same pictures:

- `E1` uses Docling's default picture-description prompt, "Describe this image in a few sentences."
- `E2` uses a prompt that names what a filing's charts carry: the type, axes, every series, and every
  value as a table, with read-off values marked. `E2` also summarises each complex table as JSON (content
  type, summary, key entities, header rows), and the summary is inlined before the table.

Three common observations about parsing diagram-heavy documents are tested here as hypotheses, and `03_replicate_carried_claims.ipynb` measures all three. Forcing OCR on born-digital pages costs a little and gains nothing: rendered at 216 dpi and read with full-page OCR, 300 pages keep 95% of the text layer's words and 98% of its numbers, and arm `T` answers 102 of 246 questions against 108 from the text layer (p = 0.18). Domain prompts do state less unsupported content (15% of descriptions against 31% for the generic prompt, judged by `gemini-3.8-flash`), but small crops do not make it worse: in these filings they are logos with little to get wrong. The diagram observation holds on 30 diagram slides from ViDoRe v3 physics, with a condition: of the relations each slide draws, the text layer states 2% and OCR 2%, while `gemini-3.5-flash-lite` states 55% when the prompt asks for arrows and connections and 18% with the generic prompt used here. This notebook measures answers with enrichment against answers without it, on filings.

In [4]:
e1 = pipe.enrichment("e1-pictures")
e2_pictures = pipe.enrichment("e2-pictures")
e2_tables = pipe.enrichment("e2-tables")
print()
costs = report.enrichment_costs(pipe)
print(f"{'pass':<13}{'elements':>9}{'in tokens':>11}{'out tokens':>11}{'s/element':>11}{'empty':>7}")
for name, row in costs.items():
    print(
        f"{name:<13}{row['elements']:>9}{row['prompt_tokens_per_element']:>11.0f}"
        f"{row['completion_tokens_per_element']:>11.0f}{row['seconds_per_element']:>11.1f}{row['empty']:>7}"
    )
audit = report.audit_summary(report.audit_descriptions(pipe).value)
print()
print("descriptions checked against their crops by Qwen3_8-Flash-Next (30 pictures, both prompts):")
for name, row in audit.items():
    print(
        f"  {name}: wrong value {row['wrong_value']:.0%}, invented label {row['invented']:.0%}, "
        f"values left out {row['missing_values']:.0%}"
    )
if SHOW_CORPUS_TEXT:
    first = sorted(e2_pictures.value)[0]
    print(first, "\nE1:", e1.value[first]["text"], "\nE2:", e2_pictures.value[first]["text"])


pass          elements  in tokens out tokens  s/element  empty
e1-pictures        117        621        176       48.5      0
e2-pictures        117        852        331       36.1      0
e2-tables          530        867        188       47.7      0
description-audit reference artifact, built with Qwen3_8-Flash-Next on Vector's cluster at $0
                  60 calls, 55,741 prompt + 1,797 completion tokens

descriptions checked against their crops by Qwen3_8-Flash-Next (30 pictures, both prompts):
  E1: wrong value 3%, invented label 3%, values left out 10%
  E2: wrong value 7%, invented label 0%, values left out 0%


A second model flagged a wrong value in 1 of 30 `E1` descriptions and 2 of 30 `E2` descriptions, and values left out in 3 of 30 `E1` descriptions and none of `E2`'s. On this vision model the generic prompt already reports most printed values; the domain prompt's gain is completeness. The seconds per element are wall time under a 20-request-per-minute cap on a shared machine, not model latency.

## Index the pages

The text arms chunk the universe with the table-aware chunker from `table_structured_reasoning` at 1,200 characters and embed the
chunks with `bge-small-en-v1.5` on the CPU. `L` runs ColBERT late interaction over `E2`'s chunks, which
separates the gain from late interaction from the gain from the image. The page-image arms use
ColModernVBERT, a 250M-parameter late-interaction model over image patches, through fastembed's ONNX
export. Its index was built on a GPU and ships through the artifact cache; the probe below re-embeds
four pages on this machine's CPU and checks that the vectors match.

In [5]:
text = pipe.text_rankings()
index = pipe.page_index()
_, query_meta = pipe.query_embeddings()
probe = pipe.cpu_probe()
print("text chunks: " + ", ".join(f"{arm} {n:,}" for arm, n in text["chunks"].items()))
print(
    f"page index: {len(index.page_ids):,} pages, {index.vectors.shape[0] / len(index.page_ids):,.0f} vectors per page, "
    f"{index.megabytes} MB in float16"
)
print(
    f"  built on {index.meta.get('hardware', 'unknown hardware')}: {index.meta['seconds_per_page']} s per page; "
    f"questions embedded at {query_meta['seconds_per_query']} s each"
)
print(
    f"  CPU probe ({probe['cpus']} vCPUs, load average {probe['load_average']}): "
    f"{probe['seconds_per_page']} s per page, {probe['seconds_per_query']} s per question"
)
print(
    f"  CPU vectors against stored vectors, lowest mean cosine: pages {probe['page_vector_cosine']}, "
    f"questions {probe['query_vector_cosine']}"
)

text chunks: T 4,595, E1 4,682, E2 4,980, L 4,980
page index: 1,000 pages, 1,119 vectors per page, 286.5 MB in float16
  built on NVIDIA GeForce RTX 4090: 0.795 s per page; questions embedded at 0.4443 s each
  CPU probe (32 vCPUs, load average 51.0): 18.14 s per page, 45.99 s per question
  CPU vectors against stored vectors, lowest mean cosine: pages 0.99999, questions 1.0


The CPU vectors match the GPU-built index to a cosine of 0.99999, so a CPU query searches the shipped index correctly. The CPU cost is the reason the index ships: about 18 s per page and 46 s per question on a shared 32-vCPU workstation, against under a second for each on the GPU. Two long FinanceBench questions ran out of the GPU's 24 GB in the query path and were embedded on the CPU in 110 s and 135 s, so query cost in this export grows steeply with question length.

## Compare page retrieval

Every arm returns ranked page identifiers. A chunk ranking collapses to pages in order of first
occurrence. `VT` and `VV` share ColModernVBERT's ranking, `TV` uses `E2`'s, and `H` fuses `E2` and `VV`
by reciprocal rank. The table reports recall@5 (`r@5`) and nDCG@5 on the gold pages, per group.

In [6]:
retrieval = report.retrieval_table(pipe, WHICH)
groups = ["figure-only", "duplicated", "table", "control", "all"]
print(f"{'arm':<5}" + "".join(f"{g:>22}" for g in groups))
print(f"{'':<5}" + "".join(f"{'r@5':>12}{'nDCG@5':>10}" for _ in groups))
for arm in ("T", "E1", "E2", "L", "VV", "H"):
    cells = "".join(f"{retrieval[arm][g]['recall']:>12.3f}{retrieval[arm][g]['ndcg']:>10.3f}" for g in groups)
    print(f"{arm:<5}{cells}")
print("n     " + "".join(f"{retrieval['T'][g]['n']:>22}" for g in groups))

arm             figure-only            duplicated                 table               control                   all
              r@5    nDCG@5         r@5    nDCG@5         r@5    nDCG@5         r@5    nDCG@5         r@5    nDCG@5
T           0.632     0.470       0.909     0.759       0.633     0.508       0.450     0.349       0.654     0.516
E1          0.921     0.732       0.886     0.753       0.650     0.514       0.450     0.339       0.754     0.605
E2          0.737     0.574       0.932     0.758       0.750     0.643       0.450     0.328       0.727     0.585
L           0.868     0.685       0.955     0.801       0.917     0.767       0.438     0.325       0.821     0.665
VV          0.882     0.770       1.000     0.903       0.767     0.636       0.375     0.254       0.782     0.666
H           0.947     0.751       0.977     0.871       0.867     0.719       0.438     0.378       0.839     0.698
n                         76                    44                    60

Page-image retrieval finds figure-only pages far better than text (recall@5 0.88 against 0.63) and loses on FinanceBench's own questions (0.38 against 0.45). Enrichment closes most of the figure gap for text retrieval, and the generic prompt does it better: `E1` descriptions reach 0.92 on figure-only pages, ahead of the domain-prompted `E2` at 0.74. A follow-up with `E2`'s picture descriptions and no table summaries reached 0.80, so the table summaries cost about 6 points by adding competing chunks, and the rest of the gap sits in the descriptions themselves: `E2` writes markdown value tables, which `bge-small-en-v1.5` matches to a question less well than `E1`'s prose. ColBERT over `E2` (`L`) recovers most of that and leads on table pages (0.92).

## Compare answers with one reader

Every row asks the same reader the same instruction; only the evidence changes. Text arms pass the top
five chunks, page arms the top three pages. The oracle rows pass the gold page (for a null, the page it
was rewritten from) as Docling text, as `E2` text, as an image, and as both. Answers count as correct
when a number in them is within 1% of the reference, or, failing that, when the judge says they state
the same thing; a null counts as correct when the reader says the evidence does not answer it.

In [7]:
pipe.show = False
accuracy = report.accuracy_table(pipe, WHICH)
pipe.show = True
readers = {row["model"] for row in accuracy.values()}
origins = {row["origin"] for row in accuracy.values()}
print(f"reader: {', '.join(sorted(readers))} ({', '.join(sorted(origins))} results)")
if len(readers) > 1:
    print("rows were answered by different models: compare only rows that share a reader")
print()
print(f"{'row':<11}" + "".join(f"{g:>13}" for g in report.GROUPS) + f"{'abstained':>11}{'in tokens':>11}{'images':>8}")
for row, a in accuracy.items():
    cells = "".join(f"{a['accuracy'][g]:>13.3f}" for g in report.GROUPS)
    print(f"{row:<11}{cells}{a['false_abstention']:>11.3f}{a['prompt_tokens']:>11,.0f}{a['images']:>8.1f}")
first = accuracy["T"]["n"]
print("n          " + "".join(f"{first[g]:>13}" for g in report.GROUPS))

reader: Qwen3_8-27B (reference results)

row          figure-only   duplicated        table      control         null  abstained  in tokens  images
T                  0.026        0.773        0.450        0.100        1.000      0.645      1,505     0.0
E1                 0.566        0.727        0.450        0.075        1.000      0.441      1,532     0.0
E2                 0.382        0.750        0.467        0.075        1.000      0.536      1,418     0.0
L                  0.553        0.795        0.517        0.125        1.000      0.441      1,628     0.0
VT                 0.855        0.955        0.683        0.175        1.000      0.232      3,350     0.0
VV                 0.842        0.932        0.683        0.200        0.846      0.218      3,393     3.0
TV                 0.618        0.818        0.717        0.275        0.846      0.273      3,401     3.0
H                  0.829        0.955        0.800        0.225        0.885      0.191      6,759     

Read the figure-only column first. Text alone answers 2 of 76. Page-image retrieval with an image reader (`VV`) answers 64, and page-image retrieval with `E2` text as the reader's input (`VT`) answers 65: once the right page is found, the domain descriptions carry the chart's values as well as the image does (`O-enriched` 0.947, `O-image` 0.934). `E2` with its own text retrieval answers only 29, so its loss is retrieval, not reading. On tables, `H` (fused retrieval, image and text) answers 0.80 against `T`'s 0.45; every text row reads the gold table perfectly (`O-text` 1.000), so that gap is also retrieval. The text rows abstain correctly on every null but also abstain on half or more of the answerable questions; the image rows answer more and abstain on nulls less (0.85 for `VV`). FinanceBench's controls stay low in every row (0.45 at best with the gold page), because many need several pages or a calculation.

## Split retrieval loss from reading loss

An arm's retrieval loss is its oracle's accuracy minus its own: what the reader would have gained had
retrieval returned the gold page. The gaps between oracle rows are reading loss: the same page, read as
text, as enriched text, or as an image.

In [8]:
loss = report.decomposition(accuracy)
print(f"{'arm':<5}{'oracle':<12}" + "".join(f"{g:>13}" for g in report.GROUPS))
for arm, row in loss.items():
    print(f"{arm:<5}{row['oracle']:<12}" + "".join(f"{row['loss'].get(g, float('nan')):>13.3f}" for g in report.GROUPS))
print()
for left, right in (("O-image", "O-text"), ("O-enriched", "O-text"), ("O-both", "O-image")):
    gaps = "  ".join(
        f"{g} {accuracy[left]['accuracy'][g] - accuracy[right]['accuracy'][g]:+.3f}" for g in report.GROUPS
    )
    print(f"{left} - {right}: {gaps}")

arm  oracle        figure-only   duplicated        table      control         null
T    O-text             -0.013        0.227        0.550        0.325        0.000
E1   O-enriched          0.382        0.273        0.550        0.350        0.000
E2   O-enriched          0.566        0.250        0.533        0.350        0.000
L    O-enriched          0.395        0.204        0.483        0.300        0.000
VT   O-enriched          0.092        0.045        0.317        0.250        0.000
VV   O-image             0.092        0.023        0.300        0.250        0.115
TV   O-image             0.316        0.136        0.267        0.175        0.115
H    O-both              0.145        0.045        0.200        0.225        0.038

O-image - O-text: figure-only +0.921  duplicated -0.045  table -0.017  control +0.025  null -0.038
O-enriched - O-text: figure-only +0.934  duplicated +0.000  table +0.000  control +0.000  null +0.000
O-both - O-image: figure-only +0.039  duplicated +0

Every text arm's loss on figure-only questions is almost all retrieval: `E2` loses 0.57 to its oracle and `E1` loses 0.38. The page-image arms lose 0.09 to 0.32. On the oracle rows, reading enriched text and reading the image differ by about one question in a hundred on figures and tables. The single large reading gap is plain text on figure-only questions, which is how those questions were selected.

## Test the differences that decide a design

Every row answers every question, so each difference is paired per question. The exact McNemar test
counts the questions one row gets right and the other wrong. This is one run with no repeats: a
difference that decides a team's design needs a validation run first.

In [9]:
print(f"{'pair':<22}{'group':<13}{'n':>5}{'left only':>11}{'right only':>12}{'p':>10}")
for row in report.paired(pipe, WHICH):
    print(
        f"{row['left'] + ' vs ' + row['right']:<22}{row['group']:<13}{row['n']:>5}{row['left_only']:>11}"
        f"{row['right_only']:>12}{row['p']:>10.4f}"
    )

pair                  group            n  left only  right only         p
VV vs T               figure-only     76         63           1    0.0000
TV vs T               figure-only     76         47           2    0.0000
E2 vs T               figure-only     76         28           1    0.0000
E2 vs E1              figure-only     76          5          19    0.0066
VT vs L               figure-only     76         27           4    0.0000
O-image vs O-text     figure-only     76         70           0    0.0000
T vs VV               table           60          5          19    0.0066
O-text vs O-image     table           60          1           0    1.0000
E2 vs VV              control         40          2           7    0.1797


With one reader, the differences that decide a design are large on this set: `VV` against `T` on figure-only questions is 63 to 1, `VT` against `L` 27 to 4, and `E1` against `E2` 19 to 5 in `E1`'s favour. On FinanceBench's controls, `E2` against `VV` (2 to 7, p = 0.18) does not separate.

## Price each approach

Costs sit in different places and compare only under a stated scenario. The reference run cost nothing
in dollars, because the on-prem models are free to the bootcamp; the paid columns price the measured
token counts at each paid family's default model's list price. Image tokens are counted differently by
each provider, so the page-image rows are estimates until the meter records your own run.

In [10]:
reader_tokens = {
    row: {"prompt_tokens": a["prompt_tokens"], "completion_tokens": a["completion_tokens"]}
    for row, a in accuracy.items()
}
per_question = report.paid_estimates(reader_tokens)
print("reader cost per 1,000 questions, US$ at list price (on-prem reference: $0)")
print(f"{'row':<11}" + "".join(f"{m:>24}" for m in report.PAID_MODELS))
for row, prices in per_question.items():
    print(f"{row:<11}" + "".join(f"{prices[m]:>24.2f}" for m in report.PAID_MODELS))
enrichment_prices = report.paid_estimates(
    {
        name: {"prompt_tokens": c["prompt_tokens_per_element"], "completion_tokens": c["completion_tokens_per_element"]}
        for name, c in costs.items()
    }
)
print()
print("enrichment cost per 1,000 elements, US$ at list price")
for name, prices in enrichment_prices.items():
    print(f"{name:<11}" + "".join(f"{prices[m]:>24.2f}" for m in report.PAID_MODELS))

reader cost per 1,000 questions, US$ at list price (on-prem reference: $0)
row                claude-haiku-5-5   gemini-3.5-flash-lite              gpt-6-luna
T                              0.16                    0.48                    0.16
E1                             0.16                    0.50                    0.16
E2                             0.15                    0.46                    0.15
L                              0.17                    0.53                    0.17
VT                             0.35                    1.06                    0.35
VV                             0.36                    1.11                    0.36
TV                             0.36                    1.11                    0.36
H                              0.70                    2.14                    0.70
O-text                         0.10                    0.33                    0.10
O-enriched                     0.14                    0.43                    0.14
O

The scenario below is a team corpus of 10,000 pages with this universe's density of pictures and
complex tables, 5,000 questions between re-indexes, and four re-indexes a year. Change the numbers to
your own. Machine time assumes the ColModernVBERT rate measured on the GPU above and Docling's parse at
about 0.35 s per page on a CPU (the `table_structured_reasoning` ingestion).

In [11]:
pages_n = len(universe.pages)
scenario = Scenario(
    pages=10_000,
    pictures_per_page=counts["pictures_described"] / pages_n,
    tables_per_page=counts["tables_summarised"] / pages_n,
    questions=5_000,
    reindexes=4,
)
machine = dict.fromkeys(arms.ARMS, 0.35)
for arm in ("VT", "VV", "H"):
    machine[arm] = 0.35 + float(index.meta["seconds_per_page"])
for model in report.PAID_MODELS:
    table = report.scenario_table(pipe, scenario, model, accuracy, machine)
    print(
        f"\n{model}: US$ {table['usd_per_element']:.5f} per enriched element; "
        f"E2 and VV cost the same at {table['break_even_questions_e2_vs_vv']:,.0f} questions a year; costs below are per year"
    )
    print(f"{'arm':<5}{'index $':>10}{'query $':>10}{'total $':>10}{'index h':>9}")
    for arm, row in table["arms"].items():
        print(
            f"{arm:<5}{row['index_usd']:>10.2f}{row['query_usd']:>10.2f}{row['total_usd']:>10.2f}{row['index_hours']:>9.1f}"
        )


claude-haiku-5-5: US$ 0.00019 per enriched element; E2 and VV cost the same at 24,098 questions a year; costs below are per year
arm     index $   query $   total $  index h
T          0.00      3.13      3.13      3.9
E1         5.01      3.20      8.21      3.9
E2         5.01      2.98      7.98      3.9
L          5.01      3.43      8.43      3.9
VT         5.01      6.91     11.91     12.7
VV         0.00      7.13      7.13     12.7
TV         5.01      7.16     12.17      3.9
H          5.01     13.98     18.98     12.7

gemini-3.5-flash-lite: US$ 0.00079 per enriched element; E2 and VV cost the same at 31,891 questions a year; costs below are per year
arm     index $   query $   total $  index h
T          0.00      9.63      9.63      3.9
E1        20.56      9.89     30.44      3.9
E2        20.56      9.21     29.77      3.9
L         20.56     10.62     31.18      3.9
VT        20.56     21.14     41.70     12.7
VV         0.00     22.10     22.10     12.7
TV        20.56

On the paid defaults, a text row costs about $0.16 per 1,000 questions on `claude-haiku-5-5` or `gpt-6-luna`, and a page row about $0.35, because three pages are about 3,400 input tokens against 1,500 for five chunks. Enrichment costs $0.15 to $0.25 per 1,000 elements on the same models.

In the scenario on `claude-haiku-5-5`, `E2` and `VV` cost about the same over a year ($7.98 against $7.13), and `E2` becomes the cheaper of the two only above about 24,000 questions a year, or about 6,000 per indexing. Dollars do not decide between them at this scale. Machine time does: `VV` needs about 12.7 machine hours a year for this corpus (four indexings of 10,000 pages, each page costing 0.35 s of parsing plus the measured GPU rate), or roughly 200 hours at the CPU probe's 18 s per page, and every live question needs a query embedding, which takes tens of seconds on a CPU. The deciding factors are retrieval quality on your questions, what an answer must cite, and where pages may be sent.

## Choose for your documents

Answer these in order on 30 to 50 of your own questions. `02_your_own_pages.ipynb` runs the first test
on your pages.

1. How many of your questions need the picture? If almost none are figure-only, text parsing with
   table-aware chunking is the default and figure work waits.
2. Are your PDFs born-digital? Use the text layer and keep OCR off. For scans, OCR quality decides the
   text families, and published results on scans conflict: test a scanned sample before choosing.
3. What is in the pictures? Dense tables favour parsed structure, with a vision model only for tables
   whose structure fails. Charts with values favour enrichment that asks for the values, or an image
   reader. Diagrams and flowcharts favour domain-prompted descriptions. Slides, infographics, and mixed
   layouts favour page-image retrieval.
4. Where can you afford to pay? Enrichment pays once per element and again at every re-index. An image
   reader pays image tokens on every question. Run the scenario cell with your page count, question
   volume, and update rate.
5. What must an answer cite? Text families cite chunks, spans, and table cells, and support payload
   filters. Page-image families cite pages.
6. What hardware and data rules apply? Page-image indexing at scale wants a GPU. Paid enrichment sends
   every picture to a provider once; a paid image reader sends retrieved pages on every question.

Scoped to what ran: one run, one vision reader (`Qwen3_8-27B`), 1,000 born-digital FinanceBench pages, and 246 questions built and checked by on-prem models. Two things would make the text families win on figures here: a retriever that matches structured descriptions (ColBERT already narrows the gap) or descriptions written as prose that names the measure, the period, and the value together. The cheapest test of the second is re-running `E2` with a prompt that writes one sentence per plotted value, which costs about 120 vision calls and a re-index.

In [12]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
